<center>
    <font size="5"> Sieci neuronowe i uczenie głębokie<br/>
        <small><em>Studia stacjonarne II stopnia 2025/2026</em><br/>Kierunek: Matematyka stosowana<br>Specjalność: Analityka danych</small>
    </font>
</center>
<br>



# Rekurencyjne Sieci Neuronowe - Zadanie

Głównym zadaniem jest stworzenie i wytrenowanie głębokiej rekurencyjnej sieci neuronowej zdolnej poprawnie rozwiązać jedno z zagadnień rozpoznawania mowy a mianowicie zagadnienie identyfikacji rozmówcy.

Dane uczące zawierają nagrania audio pochodzące z rozmowy pomiędzy osobą A i B (plik `dataset.zip` dołączony do zadania).

W trakcie realizacji zadania należy wykonań następujące podzadania:
1. Podzielić próbki na dane uczące i testowe.
2. Zdefiniować __dwa__ modele głebokiej rekurencyjnej sieci neuronowej według własnego pomysłu. W pierwszym modelu dzwięk reprezentowany powinien być za pomocą szeregu czasowego, a w drugim za pomocą spektogramu MFCC (warto wykorzystać pakiet [librosa](https://librosa.github.io/librosa/generated/librosa.feature.mfcc.html). Oba modele powinny rozwiązywać problem klasyfikacji, czy dana próbka pochodzi od osoby A czy B.
3. Wytrenować zdefiniowane sieci na danych uczących.
4. Ocenić skuteczność i porównać działanie modeli na danych testowych.
5. Najlepsze ze stworzonych modeli zapisać do pliku i przesłać na elf'a razem z notatnikiem.

__Polecam też rozszerzyć zbiór danych o nagrania własnego głosu.__

## Import bibliotek

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import librosa

print("Numpy version:", np.__version__)
print("Tensorflow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

from pathlib import Path

2026-04-15 22:08:11.594824: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Numpy version: 2.4.2
Tensorflow version: 2.20.0
Keras version: 3.13.2


## Przygotowanie danych

In [2]:
DATA_PATH = Path.cwd().joinpath("dataset")
A_PATH = DATA_PATH / "A"
B_PATH = DATA_PATH / "B"

In [ ]:
from pathlib import Path
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)


def list_audio_files(folder: Path):
    exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    return sorted(
        [p for p in folder.glob("*") if p.is_file() and p.suffix.lower() in exts]
    )


def split_one_class_3way(files, label, n_val=2, n_test=2, rng=None):
    files = np.array(files, dtype=object)
    n = len(files)

    if n < n_val + n_test + 1:
        raise ValueError(f"Za mało plików w klasie {label}: {n}")

    idx = np.arange(n)
    rng.shuffle(idx)

    val_idx = idx[:n_val]
    test_idx = idx[n_val : n_val + n_test]
    train_idx = idx[n_val + n_test :]

    X_train = files[train_idx]
    y_train = np.full(len(train_idx), label, dtype=np.int32)

    X_val = files[val_idx]
    y_val = np.full(len(val_idx), label, dtype=np.int32)

    X_test = files[test_idx]
    y_test = np.full(len(test_idx), label, dtype=np.int32)

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
import librosa
import numpy as np

SR = 16000


def load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)

    max_abs = np.max(np.abs(y))
    if max_abs > 0:
        y = y / max_abs

    return y

In [ ]:
SR = 16000
SEGMENT_SEC = 1.5
HOP_SEC = 0.75

N_FFT = 400
HOP_LENGTH = 160
N_MFCC = 13


def split_into_segments(y, sr=SR, segment_sec=SEGMENT_SEC, hop_sec=HOP_SEC):
    seg_len = int(segment_sec * sr)
    hop_len = int(hop_sec * sr)

    segments = []
    if len(y) < seg_len:
        pad = seg_len - len(y)
        y = np.pad(y, (0, pad))

    for start in range(0, len(y) - seg_len + 1, hop_len):
        segment = y[start : start + seg_len]
        segments.append(segment)

    return segments

In [12]:
a_files = list_audio_files(A_PATH)
b_files = list_audio_files(B_PATH)

Xa_train, ya_train, Xa_val, ya_val, Xa_test, ya_test = split_one_class_3way(
    a_files, label=0, n_val=2, n_test=2, rng=rng
)

Xb_train, yb_train, Xb_val, yb_val, Xb_test, yb_test = split_one_class_3way(
    b_files, label=1, n_val=2, n_test=2, rng=rng
)

X_train_paths = np.concatenate([Xa_train, Xb_train]).astype(object)
y_train = np.concatenate([ya_train, yb_train])

X_val_paths = np.concatenate([Xa_val, Xb_val]).astype(object)
y_val = np.concatenate([ya_val, yb_val])

X_test_paths = np.concatenate([Xa_test, Xb_test]).astype(object)
y_test = np.concatenate([ya_test, yb_test])

# tasowanie
train_perm = rng.permutation(len(X_train_paths))
val_perm = rng.permutation(len(X_val_paths))
test_perm = rng.permutation(len(X_test_paths))

X_train_paths, y_train = X_train_paths[train_perm], y_train[train_perm]
X_val_paths, y_val = X_val_paths[val_perm], y_val[val_perm]
X_test_paths, y_test = X_test_paths[test_perm], y_test[test_perm]

print(f"Liczba plików A: {len(a_files)}, B: {len(b_files)}")
print(f"TRAIN files: {len(X_train_paths)}")
print(f"VAL files:   {len(X_val_paths)}")
print(f"TEST files:  {len(X_test_paths)}")

Liczba plików A: 30, B: 23
TRAIN files: 45
VAL files:   4
TEST files:  4


In [13]:
def build_segment_dataset(paths, labels, feature_fn):
    X, y = [], []

    for path, label in zip(paths, labels):
        audio = load_audio(path)
        segments = split_into_segments(audio)

        for seg in segments:
            feat = feature_fn(seg)
            X.append(feat)
            y.append(label)

    return X, np.array(y, dtype=np.int32)

In [ ]:
def extract_time_features(segment, sr=SR, n_fft=400, hop_length=160):
    rms = librosa.feature.rms(y=segment, frame_length=n_fft, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(
        segment, frame_length=n_fft, hop_length=hop_length
    )[0]
    centroid = librosa.feature.spectral_centroid(
        y=segment, sr=sr, n_fft=n_fft, hop_length=hop_length
    )[0]
    bandwidth = librosa.feature.spectral_bandwidth(
        y=segment, sr=sr, n_fft=n_fft, hop_length=hop_length
    )[0]

    feats = np.stack([rms, zcr, centroid, bandwidth], axis=1)  # shape: (T, 4)
    return feats.astype(np.float32)

In [ ]:
X_train_time_list, y_train_time = build_segment_dataset(
    X_train_paths, y_train, extract_time_features
)
X_val_time_list, y_val_time = build_segment_dataset(
    X_val_paths, y_val, extract_time_features
)
X_test_time_list, y_test_time = build_segment_dataset(
    X_test_paths, y_test, extract_time_features
)

print("Model czasowy:")
print("Train segments:", len(X_train_time_list))
print("Val segments:  ", len(X_val_time_list))
print("Test segments: ", len(X_test_time_list))

Model czasowy:
Train segments: 153
Val segments:   21
Test segments:  9


In [ ]:
def extract_mfcc_features(
    segment, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH
):
    mfcc = librosa.feature.mfcc(
        y=segment, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    return mfcc.T.astype(np.float32)  # (T, n_mfcc)


X_train_mfcc_list, y_train_mfcc = build_segment_dataset(
    X_train_paths, y_train, extract_mfcc
)
X_val_mfcc_list, y_val_mfcc = build_segment_dataset(X_val_paths, y_val, extract_mfcc)
X_test_mfcc_list, y_test_mfcc = build_segment_dataset(
    X_test_paths, y_test, extract_mfcc
)

print("\nModel MFCC:")
print("Train segments:", len(X_train_mfcc_list))
print("Val segments:  ", len(X_val_mfcc_list))
print("Test segments: ", len(X_test_mfcc_list))


Model MFCC:
Train segments: 153
Val segments:   21
Test segments:  9


## Definicja i trenowanie modeli

Model czasowy:
Train segments: 164
Val segments:   8
Test segments:  11


## Ocena skuteczności